# Mutual Fund Performance Analytics

## Objective

This notebook evaluates the historical performance and risk characteristics of mutual fund schemes using:

- Daily returns
- CAGR
- Sharpe Ratio
- Sortino Ratio
- Alpha and Beta
- Maximum Drawdown
- Composite Fund Scorecard
- Benchmark comparison
- Tracking Error

### Benchmark
Nifty 100

### Risk-free rate
6.5% (RBI repo-rate proxy)

### Analysis Period
2022–2026

In [1]:
# Imports
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go

from scipy.stats import linregress

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
from pathlib import Path

PROJECT_PATH = Path(
    r"C:\Users\vansh\OneDrive\Desktop\Mutual Fund Analytics"
)

DATA_PATH = PROJECT_PATH / "data" / "processed"

print("Data path:")
print(DATA_PATH)

#file verification
print("Files available:")

for file in sorted(DATA_PATH.glob("*.csv")):
    print(" -", file.name)

Data path:
C:\Users\vansh\OneDrive\Desktop\Mutual Fund Analytics\data\processed
Files available:
 - 03_aum_by_fund_house.csv
 - 04_monthly_sip_inflows.csv
 - 05_category_inflows.csv
 - 06_industry_folio_count.csv
 - 07_scheme_performance.csv
 - 08_investor_transactions.csv
 - 09_portfolio_holdings.csv
 - 10_benchmark_indices.csv
 - fund_metadata.csv
 - nav_history.csv


In [ ]:
#loading first 3 datasets
fund_metadata = pd.read_csv(
    DATA_PATH / "fund_metadata.csv"
)

nav_history = pd.read_csv(
    DATA_PATH / "nav_history.csv"
)

benchmark = pd.read_csv(
    DATA_PATH / "10_benchmark_indices.csv"
)

print("Fund metadata:", fund_metadata.shape)
print("NAV history:", nav_history.shape)
print("Benchmark:", benchmark.shape)

Fund metadata: (35, 7)
NAV history: (97828, 3)
Benchmark: (8050, 3)


In [ ]:
# Inspecting the columns
print("Fund Metadata")
print(fund_metadata.columns.tolist())

print("\nNAV History")
print(nav_history.columns.tolist())

print("\nBenchmark")
print(benchmark.columns.tolist())

Fund Metadata
['fund_house', 'scheme_type', 'scheme_category', 'scheme_code', 'scheme_name', 'isin_growth', 'isin_div_reinvestment']

NAV History
['date', 'nav', 'scheme_code']

Benchmark
['date', 'index_name', 'close_value']


In [11]:
# cleaning NAV data
nav_history["date"] = pd.to_datetime(
    nav_history["date"],
    errors="coerce"
)

nav_history["nav"] = pd.to_numeric(
    nav_history["nav"],
    errors="coerce"
)

nav_history["scheme_code"] = pd.to_numeric(
    nav_history["scheme_code"],
    errors="coerce"
)
 # removing unusable records
nav_history = nav_history.dropna(
    subset=["date", "nav", "scheme_code"]
)

nav_history = nav_history[
    nav_history["nav"] > 0
]

# Remove duplicate scheme/date observations
nav_history = nav_history.drop_duplicates(
    subset=["date", "scheme_code"]
)

# Sorting
nav_history = nav_history.sort_values(
    ["scheme_code", "date"]
).reset_index(drop=True)

In [13]:
# Cleaning benchmark data
benchmark["date"] = pd.to_datetime(
    benchmark["date"],
    errors="coerce"
)

benchmark["close_value"] = pd.to_numeric(
    benchmark["close_value"],
    errors="coerce"
)

benchmark = benchmark.dropna(
    subset=["date", "close_value", "index_name"]
)

benchmark = benchmark[
    benchmark["close_value"] > 0
]

benchmark = benchmark.sort_values(
    ["index_name", "date"]
).reset_index(drop=True)

In [15]:
# how many schemes?
scheme_counts = (
    nav_history.groupby("scheme_code")["date"]
    .agg(
        start_date="min",
        end_date="max",
        observations="count"
    )
    .reset_index()
)

print("Unique schemes in NAV history:")
print(scheme_counts["scheme_code"].nunique())

print("\nScheme coverage:")
display(scheme_counts)

# comparing against metadata
metadata_codes = set(
    fund_metadata["scheme_code"].dropna().astype(int)
)

nav_codes = set(
    nav_history["scheme_code"].dropna().astype(int)
)

print("Schemes in metadata:", len(metadata_codes))
print("Schemes in NAV:", len(nav_codes))

print(
    "Metadata schemes missing from NAV:",
    metadata_codes - nav_codes
)

print(
    "NAV schemes missing from metadata:",
    nav_codes - metadata_codes
)

Unique schemes in NAV history:
34

Scheme coverage:


,scheme_code,start_date,end_date,observations
0,100033,2006-04-03,2026-08-04,5006
1,101206,2006-04-01,2026-08-04,7067
2,101207,2006-04-03,2006-09-11,111
3,101208,2006-04-03,2006-09-11,111
4,102885,2006-04-03,2026-08-04,4995
5,118632,2013-01-02,2026-08-04,3343
6,118633,2013-01-03,2026-08-04,3342
7,118634,2013-01-03,2026-08-04,3342
8,118636,2013-01-11,2016-11-25,1167
9,119092,2012-12-31,2026-08-04,3610


Schemes in metadata: 35
Schemes in NAV: 34
Metadata schemes missing from NAV: {0}
NAV schemes missing from metadata: set()


In [ ]:
# scheme name mapping
scheme_lookup = fund_metadata[
    ["scheme_code", "scheme_name"]
].copy()

scheme_lookup["scheme_code"] = pd.to_numeric(
    scheme_lookup["scheme_code"],
    errors="coerce"
)

scheme_lookup = scheme_lookup.drop_duplicates(
    subset="scheme_code"
)

#merging names into NAV
nav_history = nav_history.merge(
    scheme_lookup,
    on="scheme_code",
    how="left"
)

#verification
nav_history[
    ["scheme_code", "scheme_name"]
].drop_duplicates().head(20)